# Imports & Loads

In [1]:
# PATHS
TRAIN_PATH    = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\train.csv'
TEST_PATH     = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\test.csv'
SAMPLE_PATH   = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\sample_submission.csv'
ORIGINAL_PATH = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\student_health_dataset_50k.csv'

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix,
                             classification_report)
from lightgbm import LGBMClassifier

from itertools import product
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

N_FOLDS   = 5
SEED      = 42
N_CLASSES = 3

train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
original = pd.read_csv(ORIGINAL_PATH)

TARGET       = 'health_condition'
CLASS_COLORS = {'fit': '#55A868', 'at-risk': '#4C72B0', 'unhealthy': '#C44E52'}

CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']
NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']

for df in [train, test, original]:
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip().str.lower().replace('nan', np.nan)

original[TARGET] = original[TARGET].astype(str).str.strip().str.lower()
original = original.drop(columns=['student_id', 'timestamp'])

print(f'Train    : {train.shape}')
print(f'Test     : {test.shape}')
print(f'Original : {original.shape}')

Train    : (690088, 15)
Test     : (295753, 14)
Original : (50000, 14)


In [3]:
STRESS_MISSING_TR = train['stress_level'].isna().values
STRESS_MISSING_TE = test['stress_level'].isna().values
SLEEP_MISSING_TR  = train['sleep_duration'].isna().values
SLEEP_MISSING_TE  = test['sleep_duration'].isna().values

print(f'train stress missing: {STRESS_MISSING_TR.sum():,} ({STRESS_MISSING_TR.mean()*100:.1f}%)')
print(f'test  stress missing: {STRESS_MISSING_TE.sum():,} ({STRESS_MISSING_TE.mean()*100:.1f}%)')
print(f'train sleep_duration missing : {SLEEP_MISSING_TR.sum():}({SLEEP_MISSING_TR.mean()*100:.1f}%')
print(f'test Sleep_duration missng   : {SLEEP_MISSING_TE.sum():}({SLEEP_MISSING_TE.mean()*100:.1f}%)')

train stress missing: 82,811 (12.0%)
test  stress missing: 35,490 (12.0%)
train sleep_duration missing : 75999(11.0%
test Sleep_duration missng   : 32571(11.0%)


# Feature Engineering

In [4]:
from utils.feature_engineering import engineer_features


train    = engineer_features(train)
test     = engineer_features(test)
original = engineer_features(original)

ENGINEERED_CATS = ['stress_x_activity', 'stress_x_sleepq', 'stress_x_smoke','sleep_x_stress']
ENGINEERED_NUMS = ['sleep_lt6', 'sleep_ge7', 'steps_x_exercise',
                   'activity_efficiency', 'sleep_x_steps', 'bmi_band', 'sleep_dev']

FEATURES  = NUM_COLS + CAT_COLS + ENGINEERED_CATS + ENGINEERED_NUMS
MODEL_CATS = CAT_COLS + ENGINEERED_CATS
print(f'{len(FEATURES)} features ({len(MODEL_CATS)} categorical)')

24 features (10 categorical)


# Stress Level Recovery Probe

In [5]:
from utils.probes import train_probe

stress_leak_features = PROBE_LEAK = ['stress_level','stress_x_activity','stress_x_sleepq','stress_x_smoke','sleep_x_stress']

stress_probe = train_probe(train=train,target='stress_level',features = FEATURES, categorical_cols=MODEL_CATS, missing_mask= STRESS_MISSING_TR,
                           leak_features=stress_leak_features, seed=SEED,n_splits=5)

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

Probe rows: 607,277
Features: 19

Accuracy          : 0.4151 (majority baseline 0.4311)
Balanced Accuracy : 0.4068

Classification Report:
              precision    recall  f1-score   support

        high     0.3955    0.3798    0.3875    177750
         low     0.3393    0.3778    0.3575    167708
      medium     0.4850    0.4629    0.4737    261819

    accuracy                         0.4151    607277
   macro avg     0.4066    0.4068    0.4062    607277
weighted avg     0.4186    0.4151    0.4164    607277



## Training a single model for stress probe

In [ ]:
from utils.final_probe import train_final_probe
from utils.predict_proba import predict_probe

final_stress_probe = train_final_probe(train=train, target = 'stress_level',features = FEATURES, categorical_cols=MODEL_CATS, missing_mask= STRESS_MISSING_TR,
                leak_features=stress_leak_features, seed=SEED)

probs = predict_probe(train,final_stress_probe)

le = final_stress_probe['label_encoder']

for i, cls in enumerate(le.classes_):
    train[f"p_stress_{cls}"] = probs[:, i]

In [8]:
import joblib

joblib.dump(
    final_stress_probe,
    "../models/stress_probe.pkl"
)


['../models/stress_probe.pkl']

In [9]:
# Updating FEATURES and MODEL_CATS varable beacuse 3 new features just got added

FEATURES = train.columns.tolist()

MODEL_CATS = train.select_dtypes(include='object').columns.tolist()

# Sleep Duration Recovery Probe

In [11]:
# probe = train.loc[~SLEEP_MISSING_TR].reset_index(drop=True)

y_region = np.digitize(
    train["sleep_duration"],
    bins=[6.0, 7.0]
)

train_copy = train.copy()
train_copy['y_region'] = y_region

sleep_leak = ['sleep_lt6','sleep_ge7','sleep_dev','sleep_x_steps','sleep_duration']

sleep_probe_result = train_probe(
    train= train_copy,
    target = 'y_region',
    features=FEATURES,
    categorical_cols=MODEL_CATS,
    leak_features=sleep_leak,
    missing_mask= SLEEP_MISSING_TR,
    seed=SEED,
    n_splits=5   
)

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

Probe rows: 614,089
Features: 24

Accuracy          : 0.7534 (majority baseline 0.4979)
Balanced Accuracy : 0.7639

Classification Report:
              precision    recall  f1-score   support

           0     0.8400    0.8386    0.8393    134757
           1     0.5925    0.7163    0.6485    173550
           2     0.8353    0.7369    0.7830    305782

    accuracy                         0.7534    614089
   macro avg     0.7559    0.7639    0.7570    614089
weighted avg     0.7677    0.7534    0.7574    614089



In [12]:
from utils.final_probe import train_final_probe
from utils.predict_proba import predict_probe
import joblib

final_sleep_probe = train_final_probe(train=train_copy, target = 'y_region',features = FEATURES, categorical_cols=MODEL_CATS, missing_mask= SLEEP_MISSING_TR,
                leak_features=sleep_leak, seed=SEED)


joblib.dump(
    final_sleep_probe,
    "../models/sleep_probe.pkl"
)

probs = predict_probe(train,final_sleep_probe)

le = final_sleep_probe['label_encoder']

for i, cls in enumerate(le.classes_):
    train[f"p_sleep_{cls}"] = probs[:, i]


In [13]:
# Updating FEATURES and MODEL_CATS varable beacuse 3 new features just got added

FEATURES = train.columns.tolist()

MODEL_CATS = train.select_dtypes(include='object').columns.tolist()